# Netflix Data Analysis & Insights
This notebook contains the exploratory data analysis and machine learning models for the Netflix dashboard project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv('../data/netflix_titles.csv')
print(f'Dataset Shape: {df.shape}')
display(df.head())
df.info()


## 2. Data Cleaning

In [ ]:
# Handle Missing Values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])
df['duration'] = df['duration'].fillna('0')

# Date Conversions
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df['year_added'] = df['date_added'].dt.year.fillna(2020).astype(int)
df['month_added'] = df['date_added'].dt.month_name()

# Standardize Country (Take first country if multiple)
df['primary_country'] = df['country'].apply(lambda x: x.split(',')[0].strip())

# Extract Duration Numeric
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)

# Drop duplicates
df.drop_duplicates(inplace=True)
print('Data cleaning complete. Missing values remaining:')
print(df.isnull().sum())


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 1. Movies vs TV Shows
plt.figure(figsize=(8,5))
sns.countplot(x='type', data=df, palette=['#E50914', '#221f1f'])
plt.title('Distribution of Movies vs TV Shows')
plt.show()


In [ ]:
# 2. Content Added Over Years
yearly_added = df.groupby(['year_added', 'type']).size().reset_index(name='count')
px.area(yearly_added, x='year_added', y='count', color='type', title='Content Added Over Time', color_discrete_map={'Movie': '#E50914', 'TV Show': '#564d4d'})


In [ ]:
# 3. Top Countries
top_countries = df[df['primary_country'] != 'Unknown']['primary_country'].value_counts().head(10)
plt.figure(figsize=(10,6))
sns.barplot(y=top_countries.index, x=top_countries.values, palette='Reds_r')
plt.title('Top 10 Producing Countries')
plt.show()


In [ ]:
# 4. Rating Distribution
plt.figure(figsize=(12,6))
sns.countplot(x='rating', data=df, order=df['rating'].value_counts().index, palette='Reds_r')
plt.title('Content Distribution by Rating')
plt.show()


## 4. Machine Learning: Recommendation Engine

In [ ]:
# Create a combined feature string
df['combined_features'] = df['title'] + ' ' + df['director'] + ' ' + df['cast'] + ' ' + df['listed_in'] + ' ' + df['description']

# Initialize TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')


In [ ]:
def get_recommendations(title, df, tfidf_matrix):
    idx = df[df['title'].str.lower() == title.lower()].index[0]
    cosine_sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_indices = cosine_sim.argsort()[:-12:-1]
    similar_indices = [i for i in similar_indices if i != idx][:10]
    return df.iloc[similar_indices][['title', 'type', 'release_year', 'listed_in', 'primary_country']]

# Test Recommendation Engine
display(get_recommendations('Stranger Things', df, tfidf_matrix))
